# MNIST in Keras/TensorFlow — Recreated Notebook

This notebook recreates the original MNIST handwritten digit classification workflow in a cleaner, GitHub-ready form using `tensorflow.keras`.

It includes a fully connected neural network and a convolutional neural network.

In [ ]:
%matplotlib inline
import random

import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Activation, BatchNormalization, Conv2D, Dense, Dropout, Flatten, MaxPooling2D
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

## Load MNIST data

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()
print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('X_test shape:', X_test.shape)
print('y_test shape:', y_test.shape)

In [ ]:
plt.figure(figsize=(9, 9))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    idx = random.randint(0, len(X_train) - 1)
    plt.imshow(X_train[idx], cmap='gray', interpolation='none')
    plt.title(f'Class {y_train[idx]}')
    plt.axis('off')
plt.tight_layout()

## Fully connected neural network

In [ ]:
num_classes = 10

X_train_dense = X_train.reshape(60000, 784).astype('float32') / 255.0
X_test_dense = X_test.reshape(10000, 784).astype('float32') / 255.0
Y_train = to_categorical(y_train, num_classes)
Y_test = to_categorical(y_test, num_classes)

dense_model = Sequential([
    Dense(512, input_shape=(784,)),
    Activation('relu'),
    Dropout(0.2),
    Dense(512),
    Activation('relu'),
    Dropout(0.2),
    Dense(num_classes),
    Activation('softmax'),
])

dense_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
dense_model.summary()

In [ ]:
dense_model.fit(
    X_train_dense,
    Y_train,
    batch_size=128,
    epochs=5,
    verbose=1,
    validation_data=(X_test_dense, Y_test),
)
score = dense_model.evaluate(X_test_dense, Y_test, verbose=0)
print('Test score:', score[0])
print('Test accuracy:', score[1])

## Inspect predictions

In [ ]:
predicted_classes = np.argmax(dense_model.predict(X_test_dense), axis=1)
correct_indices = np.nonzero(predicted_classes == y_test)[0]
incorrect_indices = np.nonzero(predicted_classes != y_test)[0]

plt.figure(figsize=(9, 9))
for i, idx in enumerate(correct_indices[:9]):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_test[idx], cmap='gray', interpolation='none')
    plt.title(f'Predicted {predicted_classes[idx]}, Class {y_test[idx]}')
    plt.axis('off')
plt.tight_layout()

plt.figure(figsize=(9, 9))
for i, idx in enumerate(incorrect_indices[:9]):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_test[idx], cmap='gray', interpolation='none')
    plt.title(f'Predicted {predicted_classes[idx]}, Class {y_test[idx]}')
    plt.axis('off')
plt.tight_layout()

## Convolutional neural network

In [ ]:
X_train_cnn = X_train.reshape(60000, 28, 28, 1).astype('float32') / 255.0
X_test_cnn = X_test.reshape(10000, 28, 28, 1).astype('float32') / 255.0

cnn_model = Sequential()
cnn_model.add(Conv2D(32, (3, 3), input_shape=(28, 28, 1), name='conv_1'))
cnn_model.add(BatchNormalization(axis=-1))
cnn_model.add(Activation('relu', name='relu_1'))
cnn_model.add(Conv2D(32, (3, 3), name='conv_2'))
cnn_model.add(BatchNormalization(axis=-1))
cnn_model.add(Activation('relu', name='relu_2'))
cnn_model.add(MaxPooling2D(pool_size=(2, 2), name='pool_1'))
cnn_model.add(Conv2D(64, (3, 3), name='conv_3'))
cnn_model.add(BatchNormalization(axis=-1))
cnn_model.add(Activation('relu', name='relu_3'))
cnn_model.add(Conv2D(64, (3, 3), name='conv_4'))
cnn_model.add(BatchNormalization(axis=-1))
cnn_model.add(Activation('relu', name='relu_4'))
cnn_model.add(MaxPooling2D(pool_size=(2, 2), name='pool_2'))
cnn_model.add(Flatten())
cnn_model.add(Dense(512))
cnn_model.add(BatchNormalization())
cnn_model.add(Activation('relu'))
cnn_model.add(Dropout(0.2))
cnn_model.add(Dense(num_classes))
cnn_model.add(Activation('softmax'))

cnn_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
cnn_model.summary()

In [ ]:
train_datagen = ImageDataGenerator(
    rotation_range=8,
    width_shift_range=0.08,
    shear_range=0.3,
    height_shift_range=0.08,
    zoom_range=0.08,
)
test_datagen = ImageDataGenerator()

train_generator = train_datagen.flow(X_train_cnn, Y_train, batch_size=128)
test_generator = test_datagen.flow(X_test_cnn, Y_test, batch_size=128)

cnn_model.fit(
    train_generator,
    steps_per_epoch=60000 // 128,
    epochs=5,
    verbose=1,
    validation_data=test_generator,
    validation_steps=10000 // 128,
)
score = cnn_model.evaluate(X_test_cnn, Y_test, verbose=0)
print('Test score:', score[0])
print('Test accuracy:', score[1])

## Visualize convolutional activations

In [ ]:
def visualize_layer(model, layer_name, image):
    activation_model = Model(inputs=model.input, outputs=model.get_layer(layer_name).output)
    image_batch = np.expand_dims(image, axis=0)
    activations = np.squeeze(activation_model.predict(image_batch))
    num_filters = activations.shape[-1]
    grid_size = int(np.ceil(np.sqrt(num_filters)))

    plt.figure(figsize=(15, 12))
    for i in range(num_filters):
        ax = plt.subplot(grid_size, grid_size, i + 1)
        ax.imshow(activations[:, :, i], cmap='gray')
        ax.axis('off')
    plt.tight_layout()

sample_image = X_test_cnn[3]
plt.figure()
plt.imshow(sample_image.reshape(28, 28), cmap='gray', interpolation='none')
plt.axis('off')

visualize_layer(cnn_model, 'relu_1', sample_image)